In [0]:
USE CATALOG smart_odoo;

CREATE TABLE IF NOT EXISTS gold.fact_inventory
(
    -- Keys
    stock_quant_id      STRING,
    product_id          STRING,
    product_name        STRING,
 
    warehouse_id        STRING,
    warehouse_name      STRING,
 
    location_id         STRING,
    location_name       STRING,
    location_type       STRING,   -- internal / customer / supplier / view / transit
 
    company_id          STRING,
    company_name        STRING,
 
    -- Quantities
    quantity            DOUBLE,   -- on-hand qty
    reserved_quantity   DOUBLE,
    available_quantity  DOUBLE,   -- quantity - reserved_quantity
 
    -- Value  (qty * avg cost from stock_lot if available, otherwise price_unit from stock_move)
    unit_cost           DOUBLE,
    stock_value         DOUBLE,   -- available_quantity * unit_cost
 
    -- Reorder / Low Stock flags  (joined from orderpoint)
    product_min_qty     DOUBLE,   -- reorder point
    product_max_qty     DOUBLE,
    is_low_stock        BOOLEAN,  -- available_quantity < product_min_qty
 
    -- Dates
    inventory_date      TIMESTAMP,
    snapshot_date       DATE,
 
    dwh_loaded_at       TIMESTAMP
)
USING DELTA;
 
-- MERGE
 
MERGE INTO gold.fact_inventory AS target
 
USING
(
    SELECT
        sq.stock_quant_id,
        sq.product_id,
        sq.product_name,
 
        sw.warehouse_id    AS warehouse_id,
        sw.warehouse_name  AS warehouse_name,
 
        sl.stock_location_id    AS location_id,
        sl.stock_location_name  AS location_name,
        sl.usage                AS location_type,
 
        sq.company_id,
        sq.company_name,
 
        sq.quantity,
        sq.reserved_quantity,
        (sq.quantity - sq.reserved_quantity)        AS available_quantity,
 
        -- unit cost: latest price_unit from stock_move for this product
        COALESCE(sm_cost.avg_unit_cost, 0.0)        AS unit_cost,
        (sq.quantity - sq.reserved_quantity)
            * COALESCE(sm_cost.avg_unit_cost, 0.0)  AS stock_value,
 
        op.product_min_qty,
        op.product_max_qty,
        CASE
            WHEN (sq.quantity - sq.reserved_quantity) < COALESCE(op.product_min_qty, 0)
            THEN TRUE
            ELSE FALSE
        END AS is_low_stock,
 
        sq.inventory_date,
        CAST(current_date() AS DATE) AS snapshot_date,
 
        current_timestamp() AS dwh_loaded_at
 
    FROM silver.stock_quant sq
 
    -- location details
    LEFT JOIN silver.stock_location sl
        ON sl.stock_location_id = sq.location_id
 
    -- warehouse details
    LEFT JOIN silver.stock_warehouse sw
        ON sw.warehouse_id = sl.warehouse_id
 
    -- reorder rules (orderpoint) — one rule per product per location
    LEFT JOIN silver.stock_warehouse_orderpoint op
        ON  op.product_id   = sq.product_id
        AND op.location_id  = sq.location_id
        AND op.active       = TRUE
 
    -- avg unit cost from recent stock moves for this product
    LEFT JOIN
    (
        SELECT
            product_id,
            AVG(price_unit) AS avg_unit_cost
        FROM silver.stock_move
        WHERE state = 'done'
          AND price_unit > 0
        GROUP BY product_id
    ) sm_cost
        ON sm_cost.product_id = sq.product_id
 
    -- only count internal stock locations (not customers / suppliers / transit)
    WHERE sl.usage = 'internal'
      AND sq.updated_at > COALESCE
      (
          (SELECT MAX(dwh_loaded_at) FROM gold.fact_inventory),
          CAST('1900-01-01' AS TIMESTAMP)
      )
) AS source
 
ON target.stock_quant_id = source.stock_quant_id
 
WHEN MATCHED THEN
UPDATE SET
    target.product_id          = source.product_id,
    target.product_name        = source.product_name,
    target.warehouse_id        = source.warehouse_id,
    target.warehouse_name      = source.warehouse_name,
    target.location_id         = source.location_id,
    target.location_name       = source.location_name,
    target.location_type       = source.location_type,
    target.company_id          = source.company_id,
    target.company_name        = source.company_name,
    target.quantity            = source.quantity,
    target.reserved_quantity   = source.reserved_quantity,
    target.available_quantity  = source.available_quantity,
    target.unit_cost           = source.unit_cost,
    target.stock_value         = source.stock_value,
    target.product_min_qty     = source.product_min_qty,
    target.product_max_qty     = source.product_max_qty,
    target.is_low_stock        = source.is_low_stock,
    target.inventory_date      = source.inventory_date,
    target.snapshot_date       = source.snapshot_date,
    target.dwh_loaded_at       = current_timestamp()
 
WHEN NOT MATCHED THEN INSERT *;
 
 


In [0]:
Select * from gold.fact_inventory

In [0]:
USE CATALOG smart_odoo;

-- =====================================================
-- FACT: gold.fact_inventory_movement
-- =====================================================
 
CREATE TABLE IF NOT EXISTS gold.fact_inventory_movement
(
    stock_move_line_id INT,
 
    product_id INT,
    warehouse_id INT,
    lot_id INT,
    picking_id INT,
 
    from_location_id STRING,
    to_location_id STRING,
 
    quantity DOUBLE,
    unit_price DOUBLE,
    move_value DOUBLE,
 
    lot_name STRING,
    picking_name STRING,
    move_state STRING,
    picking_state STRING,
 
    is_in BOOLEAN,
    is_out BOOLEAN,
    is_internal BOOLEAN,
 
    movement_date DATE,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
 
    dwh_loaded_at TIMESTAMP
)
USING DELTA;
 
-- MERGE
 
MERGE INTO gold.fact_inventory_movement AS target
 
USING
(
    SELECT
        sml.stock_move_line_id AS stock_move_line_id,
 
        sm.product_id,
        sm.warehouse_id,
        sml.lot_id,
        sp.stock_picking_id AS picking_id,
 
        sm.location_id       AS from_location_id,
        sm.location_dest_id  AS to_location_id,
 
        CAST(sml.quantity_product_uom AS DOUBLE)                              AS quantity,
        CAST(sm.price_unit AS DOUBLE)                                         AS unit_price,
        CAST(sml.quantity_product_uom AS DOUBLE) * CAST(sm.price_unit AS DOUBLE) AS move_value,
 
        sml.lot_name,
        sp.stock_picking_name              AS picking_name,
        sm.state             AS move_state,
        sp.state             AS picking_state,
 
        CAST(sm.is_in AS BOOLEAN)  AS is_in,
        CAST(sm.is_out AS BOOLEAN) AS is_out,
        (NOT CAST(sm.is_in AS BOOLEAN) AND NOT CAST(sm.is_out AS BOOLEAN)) AS is_internal,
 
        CAST(sml.date AS DATE)        AS movement_date,
        sml.created_at,
        sml.updated_at,
 
        current_timestamp() AS dwh_loaded_at
 
    FROM silver.stock_move_line sml
    LEFT JOIN silver.stock_move sm
        ON sm.stock_move_id = sml.move_id
    LEFT JOIN silver.stock_picking sp
        ON sp.stock_picking_id = sml.picking_id
 
    WHERE sml.updated_at > COALESCE
    (
        (SELECT MAX(updated_at) FROM gold.fact_inventory_movement),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source
 
ON target.stock_move_line_id = source.stock_move_line_id
 
WHEN MATCHED AND target.updated_at < source.updated_at THEN
UPDATE SET
    target.product_id       = source.product_id,
    target.warehouse_id     = source.warehouse_id,
    target.lot_id           = source.lot_id,
    target.picking_id       = source.picking_id,
    target.from_location_id = source.from_location_id,
    target.to_location_id   = source.to_location_id,
    target.quantity         = source.quantity,
    target.unit_price       = source.unit_price,
    target.move_value       = source.move_value,
    target.lot_name         = source.lot_name,
    target.picking_name     = source.picking_name,
    target.move_state       = source.move_state,
    target.picking_state    = source.picking_state,
    target.is_in            = source.is_in,
    target.is_out           = source.is_out,
    target.is_internal      = source.is_internal,
    target.movement_date    = source.movement_date,
    target.created_at       = source.created_at,
    target.updated_at       = source.updated_at,
    target.dwh_loaded_at    = current_timestamp()
 
WHEN NOT MATCHED THEN INSERT *;
 


In [0]:
use catalog smart_odoo;

-- =====================================================
-- DIM: gold.dim_inventory_location
-- =====================================================
 
CREATE TABLE IF NOT EXISTS gold.dim_inventory_location
(
    location_id INT,
    warehouse_id INT,
 
    location_name STRING,
    warehouse_name STRING,
    location_path STRING,
    location_type STRING,
 
    is_active BOOLEAN,
 
    created_at DATE,
    updated_at DATE,
 
    dwh_loaded_at TIMESTAMP
)
USING DELTA;
 
-- MERGE
 
MERGE INTO gold.dim_inventory_location AS target
 
USING
(
    SELECT
        sl.stock_location_id    AS location_id,
        sw.warehouse_id   AS warehouse_id,
 
        sl.stock_location_name  AS location_name,
        sw.warehouse_name       AS warehouse_name,
        sl.complete_name        AS location_path,
        sl.usage                AS location_type,
 
        CAST(sl.active AS BOOLEAN) AS is_active,
 
        CAST(sl.created_at AS DATE) AS created_at,
        CAST(sl.updated_at AS DATE) AS updated_at,
 
        current_timestamp() AS dwh_loaded_at
 
    FROM silver.stock_location sl
    LEFT JOIN silver.stock_warehouse sw
        ON sw.warehouse_id = sl.warehouse_id
 
    WHERE sl.updated_at > COALESCE
    (
        (SELECT MAX(updated_at) FROM gold.dim_inventory_location),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source
 
ON target.location_id = source.location_id
 
WHEN MATCHED AND target.updated_at < source.updated_at THEN
UPDATE SET
    target.warehouse_id   = source.warehouse_id,
    target.location_name  = source.location_name,
    target.warehouse_name = source.warehouse_name,
    target.location_path  = source.location_path,
    target.location_type  = source.location_type,
    target.is_active      = source.is_active,
    target.updated_at     = source.updated_at,
    target.dwh_loaded_at  = current_timestamp()
 
WHEN NOT MATCHED THEN INSERT *;
 

In [0]:
USE CATALOG smart_odoo;
-- =====================================================
-- DIM: gold.dim_warehouse
-- =====================================================
 
CREATE TABLE IF NOT EXISTS gold.dim_warehouse
(
    warehouse_id INT,
    company_id INT,
    company_name STRING,
 
    warehouse_name STRING,
    warehouse_code STRING,
    reception_steps STRING,
    delivery_steps STRING,
 
    is_active BOOLEAN,
 
    created_at DATE,
    updated_at DATE,
 
    dwh_loaded_at TIMESTAMP
)
USING DELTA;
 
-- MERGE
 
MERGE INTO gold.dim_warehouse AS target
 
USING
(
    SELECT
        w.warehouse_id AS warehouse_id,
        w.company_id,
        w.company_name,
 
        w.warehouse_name      AS warehouse_name,
        w.code      AS warehouse_code,
        w.reception_steps,
        w.delivery_steps,
 
        CAST(w.active AS BOOLEAN) AS is_active,
 
        CAST(w.created_at AS DATE) AS created_at,
        CAST(w.updated_at AS DATE) AS updated_at,
 
        current_timestamp() AS dwh_loaded_at
 
    FROM silver.stock_warehouse w
 
    WHERE w.updated_at > COALESCE
    (
        (SELECT MAX(updated_at) FROM gold.dim_warehouse),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source
 
ON target.warehouse_id = source.warehouse_id
 
WHEN MATCHED AND target.updated_at < source.updated_at THEN
UPDATE SET
    target.company_id      = source.company_id,
    target.company_name    = source.company_name,
    target.warehouse_name  = source.warehouse_name,
    target.warehouse_code  = source.warehouse_code,
    target.reception_steps = source.reception_steps,
    target.delivery_steps  = source.delivery_steps,
    target.is_active       = source.is_active,
    target.updated_at      = source.updated_at,
    target.dwh_loaded_at   = current_timestamp()
 
WHEN NOT MATCHED THEN INSERT *;


In [0]:
USE CATALOG smart_odoo;

-- =====================================================
-- FACT: gold.fact_inventory_snapshot
-- =====================================================
drop table if exists gold.fact_inventory_snapshot;
CREATE TABLE IF NOT EXISTS gold.fact_inventory_snapshot
(
    snapshot_month DATE,

    product_id INT,
    warehouse_id INT,
    location_id INT,
    lot_id INT,

    quantity_on_hand DOUBLE,

    reserved_quantity DOUBLE,
    available_quantity DOUBLE,

    unit_cost DOUBLE,
    inventory_value DOUBLE,

    inventory_status STRING,

    is_low_stock BOOLEAN,
    is_out_of_stock BOOLEAN,

    created_at TIMESTAMP,
    updated_at TIMESTAMP,

    dwh_loaded_at TIMESTAMP
)
USING DELTA;
-- MERGE

MERGE INTO gold.fact_inventory_snapshot AS target

USING
(
    SELECT
        DATE_TRUNC('month', CURRENT_DATE()) AS snapshot_month,

        sq.product_id,
        sl.warehouse_id,
        sq.location_id,
        sq.lot_id,

        CAST(sq.quantity AS DOUBLE)
            AS quantity_on_hand,

        CAST(sq.reserved_quantity AS DOUBLE)
            AS reserved_quantity,

        CAST(
            sq.quantity - sq.reserved_quantity
            AS DOUBLE
        ) AS available_quantity,

        CAST(pp.standard_price AS DOUBLE)
            AS unit_cost,

        CAST(
            sq.quantity * pp.standard_price
            AS DOUBLE
        ) AS inventory_value,

        CASE
            WHEN sq.quantity <= 0
                THEN 'Out of Stock'

            WHEN sq.quantity < 10
                THEN 'Low Stock'

            ELSE 'In Stock'
        END AS inventory_status,

        CASE
            WHEN sq.quantity < 10
                THEN true
            ELSE false
        END AS is_low_stock,

        CASE
            WHEN sq.quantity <= 0
                THEN true
            ELSE false
        END AS is_out_of_stock,

        sq.created_at,
        sq.updated_at,

        current_timestamp() AS dwh_loaded_at

    FROM silver.stock_quant sq

    LEFT JOIN silver.stock_location sl
        ON sl.stock_location_id = sq.location_id

    LEFT JOIN silver.product_product pp
        ON pp.product_id = sq.product_id

    WHERE sq.updated_at > COALESCE
    (
        (SELECT MAX(updated_at)
         FROM gold.fact_inventory_snapshot),

        CAST('1900-01-01' AS TIMESTAMP)
    )

) AS source

ON target.product_id = source.product_id
AND target.location_id = source.location_id
AND target.snapshot_month = source.snapshot_month

WHEN MATCHED
AND target.updated_at < source.updated_at

THEN UPDATE SET

    target.warehouse_id      = source.warehouse_id,
    target.lot_id            = source.lot_id,

    target.quantity_on_hand  = source.quantity_on_hand,
    target.reserved_quantity = source.reserved_quantity,
    target.available_quantity= source.available_quantity,

    target.unit_cost         = source.unit_cost,
    target.inventory_value   = source.inventory_value,

    target.inventory_status  = source.inventory_status,

    target.is_low_stock      = source.is_low_stock,
    target.is_out_of_stock   = source.is_out_of_stock,

    target.created_at        = source.created_at,
    target.updated_at        = source.updated_at,

    target.dwh_loaded_at     = current_timestamp()

WHEN NOT MATCHED THEN INSERT *;

In [0]:
Select * from gold.fact_inventory_snapshot